# 1. Imports

In [1]:
import sqlite3
import os

# 2. Laad de sqlite en csv bestanden in

In [2]:
def create_sdm():
    db_sdm = "SDM.db"
    db_crm = "CRM-data.sqlite"
    db_gosales = "GO_SALES-data.sqlite"
    db_gostaff = "GO_STAFF-data.sqlite"

    # Verwijder het oude bestand als het al bestaat
    if os.path.exists(db_sdm):
        os.remove(db_sdm)

    print("Aanmaken van de nieuwe SDM.db database...")
    conn = sqlite3.connect(db_sdm)
    cursor = conn.cursor()

    # We zetten foreign keys tijdelijk uit tijdens het inladen van de data
    # om insert errors door 'vuile' bron data te voorkomen.
    cursor.execute("PRAGMA foreign_keys = OFF;")

    # =========================================================================
    # 1. DDL SCRIPT (Data Definition Language)
    # Bevat alle originele tabellen + Cross-Database Associaties
    # Gesorteerd op dependency order (basis tabellen eerst)
    # =========================================================================
    ddl_script = """
    -- === BASIS TABELLEN (Geen Foreign Keys) ===

    CREATE TABLE "age_group" (
        [AGE_GROUP_CODE] TEXT, [UPPER_AGE] TEXT, [LOWER_AGE] TEXT,
        PRIMARY KEY ([AGE_GROUP_CODE])
    );

    CREATE TABLE "customer_segment" (
        [SEGMENT_CODE] TEXT, [LANGUAGE] TEXT, [SEGMENT_NAME] TEXT, [SEGMENT_DESCRIPTION] TEXT,
        PRIMARY KEY ([SEGMENT_CODE])
    );

    CREATE TABLE "customer_type" (
        [CUSTOMER_TYPE_CODE] TEXT, [CUSTOMER_TYPE_EN] TEXT,
        PRIMARY KEY ([CUSTOMER_TYPE_CODE])
    );

    CREATE TABLE "sales_territory" (
        [SALES_TERRITORY_CODE] TEXT, [TERRITORY_NAME_EN] TEXT,
        PRIMARY KEY ([SALES_TERRITORY_CODE])
    );

    CREATE TABLE "country" (
        [COUNTRY_CODE] TEXT, [COUNTRY] TEXT, [LANGUAGE] TEXT, [CURRENCY_NAME] TEXT,
        PRIMARY KEY ([COUNTRY_CODE])
    );

    CREATE TABLE "order_method" (
        [ORDER_METHOD_CODE] TEXT, [ORDER_METHOD_EN] TEXT,
        PRIMARY KEY ([ORDER_METHOD_CODE])
    );

    CREATE TABLE "product_line" (
        [PRODUCT_LINE_CODE] TEXT, [PRODUCT_LINE_EN] TEXT,
        PRIMARY KEY ([PRODUCT_LINE_CODE])
    );

    CREATE TABLE "return_reason" (
        [RETURN_REASON_CODE] TEXT, [RETURN_DESCRIPTION_EN] TEXT,
        PRIMARY KEY ([RETURN_REASON_CODE])
    );

    CREATE TABLE "course" (
        [COURSE_CODE] TEXT, [COURSE_DESCRIPTION] TEXT,
        PRIMARY KEY ([COURSE_CODE])
    );

    CREATE TABLE "satisfaction_type" (
        [SATISFACTION_TYPE_CODE] TEXT, [SATISFACTION_TYPE_DESCRIPTION] TEXT,
        PRIMARY KEY ([SATISFACTION_TYPE_CODE])
    );

    -- === NIVEAU 1 TABELLEN ===

    CREATE TABLE "crm_country" (
        "COUNTRY_CODE" TEXT, "COUNTRY_EN" TEXT, "FLAG_IMAGE" TEXT, "SALES_TERRITORY_CODE" TEXT,
        PRIMARY KEY("COUNTRY_CODE"),
        FOREIGN KEY("SALES_TERRITORY_CODE") REFERENCES "sales_territory"("SALES_TERRITORY_CODE")
    );

    CREATE TABLE "customer_headquarters" (
        "CUSTOMER_CODEMR" TEXT, "CUSTOMER_NAME" TEXT, "ADDRESS1" TEXT, "ADDRESS2" TEXT,
        "CITY" TEXT, "REGION" TEXT, "POSTAL_ZONE" TEXT, "COUNTRY_CODE" TEXT,
        "PHONE" TEXT, "FAX" TEXT, "SEGMENT_CODE" TEXT,
        PRIMARY KEY("CUSTOMER_CODEMR"),
        FOREIGN KEY("SEGMENT_CODE") REFERENCES "customer_segment"("SEGMENT_CODE")
    );

    CREATE TABLE "product_type" (
        [PRODUCT_TYPE_CODE] TEXT, [PRODUCT_LINE_CODE] TEXT, [PRODUCT_TYPE_EN] TEXT,
        PRIMARY KEY ([PRODUCT_TYPE_CODE]),
        FOREIGN KEY ([PRODUCT_LINE_CODE]) REFERENCES [product_line]([PRODUCT_LINE_CODE]) ON DELETE RESTRICT
    );

    CREATE TABLE "sales_branch" (
        "SALES_BRANCH_CODE" TEXT, "ADDRESS1" TEXT, "ADDRESS2" TEXT, "CITY" TEXT,
        "REGION" TEXT, "POSTAL_ZONE" TEXT, "COUNTRY_CODE" TEXT,
        PRIMARY KEY("SALES_BRANCH_CODE"),
        FOREIGN KEY("COUNTRY_CODE") REFERENCES "country"("COUNTRY_CODE") ON DELETE RESTRICT
    );

    CREATE TABLE "sales_office" (
        [SALES_OFFICE_CODE] TEXT, [STREET] TEXT, [ADDITION] TEXT, [CITY] TEXT,
        [REGION] TEXT, [ZIPCODE] TEXT, [COUNTRY_CODE] TEXT,
        PRIMARY KEY ([SALES_OFFICE_CODE]),
        -- CROSS-DB FK:
        FOREIGN KEY([COUNTRY_CODE]) REFERENCES "country"("COUNTRY_CODE")
    );

    -- === NIVEAU 2 TABELLEN ===

    CREATE TABLE "customer" (
        "CUSTOMER_CODE" TEXT, "CUSTOMER_CODEMR" TEXT, "COMPANY_NAME" TEXT, "CUSTOMER_TYPE_CODE" TEXT,
        PRIMARY KEY("CUSTOMER_CODE"),
        FOREIGN KEY("CUSTOMER_CODEMR") REFERENCES "customer_headquarters"("CUSTOMER_CODEMR"),
        FOREIGN KEY("CUSTOMER_TYPE_CODE") REFERENCES "customer_type"("CUSTOMER_TYPE_CODE")
    );

    CREATE TABLE "sales_demographic" (
        "DEMOGRAPHIC_CODE" TEXT, "CUSTOMER_CODEMR" TEXT, "AGE_GROUP_CODE" TEXT, "SALES_PERCENT" TEXT,
        FOREIGN KEY("AGE_GROUP_CODE") REFERENCES "age_group"("AGE_GROUP_CODE") ON DELETE CASCADE,
        FOREIGN KEY("CUSTOMER_CODEMR") REFERENCES "customer_headquarters"("CUSTOMER_CODEMR") ON DELETE CASCADE
    );

    CREATE TABLE "product" (
        [PRODUCT_NUMBER] TEXT, [INTRODUCTION_DATE] TEXT, [PRODUCT_TYPE_CODE] TEXT, [PRODUCTION_COST] TEXT,
        [MARGIN] TEXT, [PRODUCT_IMAGE] TEXT, [LANGUAGE] TEXT, [PRODUCT_NAME] TEXT, [DESCRIPTION] TEXT,
        PRIMARY KEY ([PRODUCT_NUMBER]),
        FOREIGN KEY ([PRODUCT_TYPE_CODE]) REFERENCES [product_type]([PRODUCT_TYPE_CODE]) ON DELETE RESTRICT
    );

    CREATE TABLE "sales_staff" (
        [SALES_STAFF_CODE] TEXT, [FIRST_NAME] TEXT, [LAST_NAME] TEXT, [POSITION_EN] TEXT, [WORK_PHONE] TEXT,
        [EXTENSION] TEXT, [FAX] TEXT, [EMAIL] TEXT, [DATE_HIRED] TEXT, [SALES_BRANCH_CODE] TEXT,
        PRIMARY KEY ([SALES_STAFF_CODE]),
        FOREIGN KEY ([SALES_BRANCH_CODE]) REFERENCES [sales_branch]([SALES_BRANCH_CODE]) ON DELETE RESTRICT
    );

    CREATE TABLE "sales_representative" (
        "SALES_REPRESENTATIVE_CODE" TEXT, "FIRST_NAME" TEXT, "LAST_NAME" TEXT, "POSITION_EN" TEXT,
        "WORK_PHONE" TEXT, "EXTENSION" TEXT, "FAX" TEXT, "EMAIL" TEXT, "DATE_HIRED" TEXT,
        "SALES_OFFICE_CODE" TEXT, "MANAGER_CODE" TEXT,
        PRIMARY KEY("SALES_REPRESENTATIVE_CODE"),
        FOREIGN KEY("MANAGER_CODE") REFERENCES "sales_representative"("SALES_REPRESENTATIVE_CODE"),
        FOREIGN KEY("SALES_OFFICE_CODE") REFERENCES "sales_office"("SALES_OFFICE_CODE")
    );

    -- === NIVEAU 3 TABELLEN ===

    CREATE TABLE "customer_store" (
        "CUSTOMER_SITE_CODE" TEXT, "CUSTOMER_CODE" TEXT, "STREET" TEXT, "ADDITION" TEXT,
        "CITY" TEXT, "STATE" TEXT, "ZIPCODE" TEXT, "COUNTRY_CODE" TEXT, "ACTIVE_INDICATOR" TEXT,
        PRIMARY KEY("CUSTOMER_SITE_CODE"),
        FOREIGN KEY("COUNTRY_CODE") REFERENCES "crm_country"("COUNTRY_CODE"),
        FOREIGN KEY("CUSTOMER_CODE") REFERENCES "customer"("CUSTOMER_CODE")
    );

    CREATE TABLE "retailer_site" (
        [RETAILER_SITE_CODE] TEXT, [RETAILER_CODE] TEXT, [ADDRESS1] TEXT, [ADDRESS2] TEXT,
        [CITY] TEXT, [REGION] TEXT, [POSTAL_ZONE] TEXT, [COUNTRY_CODE] TEXT, [ACTIVE_INDICATOR] TEXT,
        PRIMARY KEY ([RETAILER_SITE_CODE]),
        -- CROSS-DB FKs:
        FOREIGN KEY([RETAILER_CODE]) REFERENCES "customer"("CUSTOMER_CODE"),
        FOREIGN KEY([COUNTRY_CODE]) REFERENCES "country"("COUNTRY_CODE")
    );

    CREATE TABLE "satisfaction" (
        "YEAR" TEXT, "SALES_REPRESENTATIVE_CODE" TEXT, "SATISFACTION_TYPE_CODE" TEXT,
        FOREIGN KEY ([SALES_REPRESENTATIVE_CODE]) REFERENCES [sales_representative]([SALES_REPRESENTATIVE_CODE]) ON DELETE CASCADE,
        FOREIGN KEY ([SATISFACTION_TYPE_CODE]) REFERENCES [satisfaction_type]([SATISFACTION_TYPE_CODE]) ON DELETE CASCADE
    );

    CREATE TABLE "training" (
        "YEAR" TEXT, "SALES_REPRESENTATIVE_CODE" TEXT, "COURSE_CODE" TEXT,
        FOREIGN KEY ([COURSE_CODE]) REFERENCES [course]([COURSE_CODE]) ON DELETE CASCADE,
        FOREIGN KEY ([SALES_REPRESENTATIVE_CODE]) REFERENCES [sales_representative]([SALES_REPRESENTATIVE_CODE]) ON DELETE CASCADE
    );

    -- === NIVEAU 4 TABELLEN ===

    CREATE TABLE "customer_contact" (
        [CUSTOMER_CONTACT_CODE] TEXT, [CUSTOMER_SITE_CODE] TEXT, [FIRST_NAME] TEXT, [LAST_NAME] TEXT,
        [JOB_POSITION_EN] TEXT, [EXTENSION] TEXT, [FAX] TEXT, [E_MAIL] TEXT, [GENDER] TEXT,
        PRIMARY KEY ([CUSTOMER_CONTACT_CODE]),
        -- IMPLICIETE FK TOEGEVOEGD:
        FOREIGN KEY ([CUSTOMER_SITE_CODE]) REFERENCES "customer_store"("CUSTOMER_SITE_CODE")
    );

    -- === NIVEAU 5 TABELLEN ===

    CREATE TABLE "order_header" (
        [ORDER_NUMBER] TEXT, [RETAILER_NAME] TEXT, [RETAILER_SITE_CODE] TEXT, [RETAILER_CONTACT_CODE] TEXT,
        [SALES_STAFF_CODE] TEXT, [SALES_BRANCH_CODE] TEXT, [ORDER_DATE] TEXT, [ORDER_METHOD_CODE] TEXT,
        PRIMARY KEY ([ORDER_NUMBER]),
        FOREIGN KEY ([ORDER_METHOD_CODE]) REFERENCES [order_method]([ORDER_METHOD_CODE]) ON DELETE RESTRICT,
        FOREIGN KEY ([RETAILER_SITE_CODE]) REFERENCES [retailer_site]([RETAILER_SITE_CODE]) ON DELETE RESTRICT,
        FOREIGN KEY ([SALES_BRANCH_CODE]) REFERENCES [sales_branch]([SALES_BRANCH_CODE]) ON DELETE RESTRICT,
        FOREIGN KEY ([SALES_STAFF_CODE]) REFERENCES [sales_staff]([SALES_STAFF_CODE]) ON DELETE RESTRICT,
        -- CROSS-DB FK:
        FOREIGN KEY ([RETAILER_CONTACT_CODE]) REFERENCES "customer_contact"("CUSTOMER_CONTACT_CODE")
    );

    -- === NIVEAU 6 TABELLEN ===

    CREATE TABLE "order_details" (
        [ORDER_DETAIL_CODE] TEXT, [ORDER_NUMBER] TEXT, [PRODUCT_NUMBER] TEXT,
        [QUANTITY] TEXT, [UNIT_COST] TEXT, [UNIT_PRICE] TEXT, [UNIT_SALE_PRICE] TEXT,
        PRIMARY KEY ([ORDER_DETAIL_CODE]),
        FOREIGN KEY ([ORDER_NUMBER]) REFERENCES [order_header]([ORDER_NUMBER]) ON DELETE RESTRICT,
        FOREIGN KEY ([PRODUCT_NUMBER]) REFERENCES [product]([PRODUCT_NUMBER]) ON DELETE RESTRICT
    );

    -- === NIVEAU 7 TABELLEN ===

    CREATE TABLE "returned_item" (
        [RETURN_CODE] TEXT, [RETURN_DATE] TEXT, [ORDER_DETAIL_CODE] TEXT, [RETURN_REASON_CODE] TEXT, [RETURN_QUANTITY] TEXT,
        PRIMARY KEY ([RETURN_CODE]),
        FOREIGN KEY ([ORDER_DETAIL_CODE]) REFERENCES [order_details]([ORDER_DETAIL_CODE]) ON DELETE RESTRICT,
        FOREIGN KEY ([RETURN_REASON_CODE]) REFERENCES [return_reason]([RETURN_REASON_CODE]) ON DELETE RESTRICT
    );
    """

    # Voer het schema uit in de nieuwe database
    cursor.executescript(ddl_script)
    print("Schema succesvol aangemaakt.")

    # =========================================================================
    # 2. DATA INLADEN VANUIT DE BRON DATABASES
    # =========================================================================
    print("Koppelen van de bron databases...")
    cursor.execute(f"ATTACH DATABASE '{db_crm}' AS crm;")
    cursor.execute(f"ATTACH DATABASE '{db_gosales}' AS gosales;")
    cursor.execute(f"ATTACH DATABASE '{db_gostaff}' AS gostaff;")

    # Lijst van (bron_alias, tabel_naam)
    tabellen_mapping = [
        # CRM
        ("crm", "age_group"), ("crm", "customer_segment"), ("crm", "customer_type"),
        ("crm", "sales_territory"), ("crm", "crm_country"), ("crm", "customer_headquarters"),
        ("crm", "customer"), ("crm", "sales_demographic"), ("crm", "customer_store"),
        ("crm", "customer_contact"),
        # GO_SALES
        ("gosales", "country"), ("gosales", "order_method"), ("gosales", "product_line"),
        ("gosales", "return_reason"), ("gosales", "product_type"), ("gosales", "sales_branch"),
        ("gosales", "product"), ("gosales", "sales_staff"), ("gosales", "retailer_site"),
        ("gosales", "order_header"), ("gosales", "order_details"), ("gosales", "returned_item"),
        # GO_STAFF
        ("gostaff", "course"), ("gostaff", "satisfaction_type"), ("gostaff", "sales_office"),
        ("gostaff", "sales_representative"), ("gostaff", "satisfaction"), ("gostaff", "training")
    ]

    print("Data kopiëren naar het Source Data Model...")
    for db_alias, table in tabellen_mapping:
        try:
            cursor.execute(f"INSERT INTO main.{table} SELECT * FROM {db_alias}.{table};")
            # print(f"  - Data voor '{table}' succesvol gekopieerd.")
        except sqlite3.Error as e:
            print(f"Fout bij het kopiëren van '{table}' uit '{db_alias}': {e}")

    conn.commit()
    conn.close()

    print("\nProces afgerond! Het bestand 'SDM.db' is met succes gegenereerd.")

if __name__ == "__main__":
    create_sdm()

Aanmaken van de nieuwe SDM.db database...
Schema succesvol aangemaakt.
Koppelen van de bron databases...
Data kopiëren naar het Source Data Model...

Proces afgerond! Het bestand 'SDM.db' is met succes gegenereerd.
